In [ ]:
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from PIL import Image
from pathlib import Path

project_root = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / 'dsl' / 'executor.py').is_file())
sys.path.append(str(project_root))

CLEVR_COLORS = {
    'red': '#FF4444', 'blue': '#4499FF', 'green': '#44CC44',
    'yellow': '#FFCC00', 'cyan': '#00CCDD', 'purple': '#AA44CC',
    'brown': '#AA6633', 'gray': '#999999',
}
TARGET_IMAGE = 'CLEVR_val_000138'

clevr_path = project_root / 'datasets' / 'converted' / 'clevr' / 'clevr_val.jsonl'
with open(clevr_path) as f:
    for line in f:
        d = json.loads(line)
        if d['image_id'] == TARGET_IMAGE:
            sample = d
            break

image_path = project_root / sample['image_path']
img_np = np.array(Image.open(image_path).convert('RGB'))
print(f'Слика: {TARGET_IMAGE}  ({img_np.shape[1]}×{img_np.shape[0]} px)')


## 1. Детекција објеката — YOLOv8 + FastSAM

In [ ]:
from scene_graph.builder import SceneGraphBuilder
from scene_graph.relations import RelationConfig

builder = SceneGraphBuilder(relation_config=RelationConfig())
scene_graph, intermediates = builder.build(
    image_path,
    use_segmentation=True,
    use_depth=True,
    use_heuristics=True,
    return_intermediates=True,
)
detections = intermediates.get('detections', [])
segments   = intermediates.get('segments', [])
depth_map  = intermediates.get('depth_map')
print(f'Детектовано {len(detections)} објекта')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].imshow(img_np)
axes[0].axis('off')
axes[0].set_title('Оригинална слика', fontsize=13, pad=8)

axes[1].imshow(img_np)
axes[1].axis('off')
axes[1].set_title('Детектовани објекти (YOLOv8 + FastSAM)', fontsize=13, pad=8)

cmap = plt.get_cmap('tab10')
for i, det in enumerate(detections):
    color = cmap(i % 10)
    x1, y1, x2, y2 = det['bbox']
    w, h = x2 - x1, y2 - y1
    axes[1].add_patch(mpatches.FancyBboxPatch(
        (x1, y1), w, h,
        boxstyle='round,pad=2', linewidth=0,
        edgecolor=color, facecolor=color, alpha=0.15, zorder=3,
    ))
    axes[1].add_patch(mpatches.FancyBboxPatch(
        (x1, y1), w, h,
        boxstyle='round,pad=2', linewidth=4,
        edgecolor=color, facecolor='none', alpha=0.95, zorder=4,
    ))
    conf = det.get('confidence', 0)
    axes[1].text(
        x1 + 4, y1 - 4,
        f"Сигурност: {conf*100:.0f}%",
        ha='left', va='bottom', fontsize=8, color='white', weight='bold',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.6, lw=0),
        zorder=8,
    )
    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2
    axes[1].plot(cx, cy, 'o', markersize=16, color=color,
                 markeredgecolor='white', markeredgewidth=1.5, zorder=6)
    axes[1].text(cx, cy, str(i), ha='center', va='center',
                 fontsize=9, weight='bold', color='white', zorder=7)

plt.tight_layout(pad=1.5)
plt.show()


## 2. Процена дубине — MiDaS

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].imshow(img_np)
axes[0].axis('off')
axes[0].set_title('Оригинална слика', fontsize=13, pad=8)

if depth_map is not None:
    dm = depth_map.astype(float)
    dm_norm = (dm - dm.min()) / (dm.max() - dm.min() + 1e-8)
    im = axes[1].imshow(dm_norm, cmap='plasma')
    axes[1].axis('off')
    axes[1].set_title('Процењена дубина (MiDaS)', fontsize=13, pad=8)
    cbar = fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
    cbar.set_label('Релативна дубина (MiDaS: веће вредности = ближе)', fontsize=8)
    cbar.set_ticks([0, 0.5, 1])
    cbar.set_ticklabels(['далеко', 'средина', 'близу'], fontsize=8)
else:
    axes[1].text(0.5, 0.5, 'Нема карте дубине', ha='center', va='center',
                 fontsize=12, transform=axes[1].transAxes)
    axes[1].axis('off')

plt.tight_layout(pad=1.5)
plt.show()


## 3. Сегментација — FastSAM маске

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].imshow(img_np)
axes[0].axis('off')
axes[0].set_title('Оригинална слика', fontsize=13, pad=8)

axes[1].imshow(img_np, alpha=0.45)
axes[1].axis('off')
axes[1].set_title('Обрађене маске након примене Z-бафера', fontsize=13, pad=8)

cmap_seg = plt.get_cmap('tab10')
if segments:
    for i, seg in enumerate(segments):
        mask = seg.get('mask')
        if mask is None:
            continue
        color = np.array(cmap_seg(i % 10)[:3])
        overlay = np.zeros((*mask.shape, 4), dtype=float)
        overlay[mask > 0] = [*color, 0.55]
        axes[1].imshow(overlay, zorder=3)
        ys, xs = np.where(mask > 0)
        if len(xs):
            cx, cy = xs.mean(), ys.mean()
            axes[1].plot(cx, cy, 'o', markersize=14, color=color,
                         markeredgecolor='white', markeredgewidth=1.5, zorder=5)
            axes[1].text(cx, cy, str(i), ha='center', va='center',
                         fontsize=8, weight='bold', color='white', zorder=6)
else:
    axes[1].text(0.5, 0.5, 'Нема маски', ha='center', va='center',
                 fontsize=12, transform=axes[1].transAxes)

plt.tight_layout(pad=1.5)
plt.show()
